# 11b -- Quantum Error Correction

**Prerequisites:** Notebook 11. Familiarity with noise channels and density matrices.

Quantum error correction (QEC) is the path to **fault-tolerant quantum
computing**. In notebook 11 we saw how noise corrupts quantum states, and
in notebook 15 we will explore error *mitigation* -- classical post-processing
tricks that improve noisy expectation values. But mitigation is a bandage,
not a cure: it cannot fix individual shots or scale to arbitrary circuit
depth.

QEC is the cure. It encodes a single **logical qubit** into several
**physical qubits** in such a way that errors can be detected and
corrected *during the computation* -- without ever learning (or
disturbing) the encoded quantum information.

By the end of this notebook you will be able to:

1. **Describe** why quantum error correction is fundamentally harder than classical error correction.
2. **Implement** the 3-qubit bit-flip and phase-flip repetition codes.
3. **Explain** syndrome measurement and how it detects errors without disturbing the encoded state.
4. **Demonstrate** the Shor 9-qubit code that corrects arbitrary single-qubit errors.

In this notebook we will:

1. Understand why naive classical strategies fail for quantum states.
2. Build the **3-qubit bit-flip code** -- the simplest QEC code.
3. Perform **syndrome measurement** to detect errors without collapsing the logical qubit.
4. Build the **3-qubit phase-flip code** using the Hadamard basis.
5. Combine both into the **Shor 9-qubit code** that corrects arbitrary single-qubit errors.
6. Preview the **stabilizer formalism** and connect to Clifford simulation (notebook 16).

### Misconception: "Just copy the qubit 3 times like a classical repetition code"

Classical error correction works by copying bits: store three copies,
take a majority vote. But the **no-cloning theorem** forbids copying an
unknown quantum state. And even if we could copy, **measurement destroys
superposition** -- we cannot peek at individual qubits to check for
errors without collapsing the encoded state. Quantum error correction
must extract error information (the *syndrome*) without learning
anything about the encoded data. This is a fundamentally harder problem.

In [1]:
import (
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/sim/clifford"
	"github.com/splch/goqu/sim/densitymatrix"
	"github.com/splch/goqu/sim/noise"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## Why Quantum Error Correction Is Hard

Classical error correction is straightforward: copy a bit three times
($0 \to 000$, $1 \to 111$), and if one bit flips, majority vote
recovers the original. Two things make this impossible for qubits:

1. **The no-cloning theorem.** There is no quantum operation that takes
   an unknown state $|\psi\rangle$ and produces $|\psi\rangle \otimes |\psi\rangle$.
   If we try to "copy" using CNOT, we get *entanglement*, not independent copies.

2. **Measurement collapses the state.** In classical EC, we can freely
   inspect each bit to find the error. In quantum EC, measuring a qubit
   destroys its superposition. We need a way to extract error information
   (the **syndrome**) without learning anything about the encoded state.

Let's see the no-cloning theorem in action. We will try to "copy"
$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ by applying CNOT
with the data qubit as control and a fresh $|0\rangle$ as target.

In [2]:
%%
// Attempt to "clone" |psi> = RY(pi/3)|0> using CNOT.
// If cloning worked, we'd get |psi> x |psi>.
// Instead, CNOT creates an entangled state.

cClone, err := builder.New("clone-attempt", 2).
	RY(math.Pi/3, 0). // Prepare |psi> = cos(pi/6)|0> + sin(pi/6)|1>
	CNOT(0, 1).        // "Copy" attempt
	Build()
if err != nil {
	panic(err)
}

fmt.Println("Naive cloning attempt: CNOT(data, target)")
gonbui.DisplayHTML(draw.SVG(cClone))

sim := statevector.New(2)
if err := sim.Evolve(cClone); err != nil {
	panic(err)
}

sv := sim.StateVector()
fmt.Println("Result statevector:")
for i, amp := range sv {
	p := real(amp)*real(amp) + imag(amp)*imag(amp)
	if p > 1e-10 {
		fmt.Printf("  |%02b> : %.4f  (prob = %.4f)\n", i, amp, p)
	}
}

// If cloning worked, we'd have |psi> x |psi> = (cos^2)|00> + cos*sin|01> + cos*sin|10> + sin^2|11>
// Instead we get an ENTANGLED state: cos|00> + sin|11>
fmt.Println("\nIf cloning worked, we would see amplitudes on |01> and |10>.")
fmt.Println("Instead, CNOT creates the entangled state cos(pi/6)|00> + sin(pi/6)|11>.")
fmt.Println("This is entanglement, not copying. The no-cloning theorem holds.")
fmt.Println("\nBut wait -- this entangled encoding is actually USEFUL!")
fmt.Println("It is the starting point for quantum error correction.")

Naive cloning attempt: CNOT(data, target)
Result statevector:
  |00> : (0.8660+0.0000i)  (prob = 0.7500)
  |11> : (0.5000+0.0000i)  (prob = 0.2500)

If cloning worked, we would see amplitudes on |01> and |10>.
Instead, CNOT creates the entangled state cos(pi/6)|00> + sin(pi/6)|11>.
This is entanglement, not copying. The no-cloning theorem holds.

But wait -- this entangled encoding is actually USEFUL!
It is the starting point for quantum error correction.


q0 
 
 q1 
 
 RY(pi/3)

## The 3-Qubit Bit-Flip Code

The simplest quantum error-correcting code protects against **bit-flip
errors** (X errors). It uses three physical qubits to encode one logical
qubit.

### Encoding

We encode the logical states as:

$$|0\rangle_L = |000\rangle, \qquad |1\rangle_L = |111\rangle$$

A general state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ becomes:

$$|\psi\rangle_L = \alpha|000\rangle + \beta|111\rangle$$

The encoding circuit is simple: apply CNOT from the data qubit to two
ancilla qubits initialized to $|0\rangle$:

```
q0 (data):    --|-----*-----*--
q1 (ancilla): --|-----|--X--|--
q2 (ancilla): --|-----|-----|--X--
                      CNOT  CNOT
```

This creates entanglement, not copies. The three qubits are in a
joint state that cannot be factored -- and that is exactly what
protects the information.

### Error and Syndrome Measurement

Suppose a bit-flip error (X gate) hits one of the three data qubits.
We need to figure out *which* qubit was flipped without measuring the
data qubits directly (that would destroy the superposition).

The trick is **syndrome measurement**: we use two ancilla qubits to
measure the *parity* of pairs of data qubits. The parity tells us
whether two qubits agree or disagree, without revealing their values:

| Syndrome (s1 s0) | Meaning | Error location |
|:-:|:---|:---|
| 00 | All parities agree | No error |
| 01 | q0 and q1 disagree | Error on q0 |
| 11 | q0-q1 and q1-q2 both disagree | Error on q1 |
| 10 | q1 and q2 disagree | Error on q2 |

We then apply a corrective X gate to the identified qubit.

In [3]:
%%
// 3-qubit bit-flip code: encode, inject error, syndrome extraction, correction.
// Qubits: q0-q2 = data qubits, q3-q4 = syndrome ancillae
// Classical bits: c0-c1 = syndrome results

cBitFlip, err := builder.New("bit-flip-code", 5).WithClbits(2).
	// Prepare logical qubit in superposition: |psi> = H|0> = |+>
	H(0).
	// Encode: |psi> -> alpha|000> + beta|111>
	CNOT(0, 1).CNOT(0, 2).
	// Inject error: bit flip on qubit 1
	X(1).
	// Syndrome extraction: check parity of qubit pairs
	CNOT(0, 3).CNOT(1, 3). // ancilla q3 = q0 XOR q1
	CNOT(1, 4).CNOT(2, 4). // ancilla q4 = q1 XOR q2
	// Measure syndromes
	Measure(3, 0). // q3 -> c0 (s0)
	Measure(4, 1). // q4 -> c1 (s1)
	// Correction: use Switch on syndrome bits [c0, c1]
	// Switch([]int{0, 1}, ...) reads clbits [c0, c1].
	// The integer value = c0 + 2*c1:
	//   c0=1, c1=0 -> value 1 -> error on q0
	//   c0=0, c1=1 -> value 2 -> error on q2
	//   c0=1, c1=1 -> value 3 -> error on q1
	//   c0=0, c1=0 -> value 0 -> no error
	Switch([]int{0, 1}, map[int]func(*builder.Builder){
		1: func(b *builder.Builder) { b.X(0) }, // c0=1,c1=0 -> fix q0
		2: func(b *builder.Builder) { b.X(2) }, // c0=0,c1=1 -> fix q2
		3: func(b *builder.Builder) { b.X(1) }, // c0=1,c1=1 -> fix q1
	}, nil).
	Build()
if err != nil {
	panic(err)
}

fmt.Println("3-Qubit Bit-Flip Code (error on q1):")
gonbui.DisplayHTML(draw.SVG(cBitFlip))

// Run with dynamic circuit support (needed for mid-circuit measurement + Switch)
simBF := statevector.New(5)
counts, err := simBF.RunDynamic(cBitFlip, 1000)
if err != nil {
	panic(err)
}

fmt.Println("\nMeasurement results (1000 shots):")
for bs, n := range counts {
	fmt.Printf("  %s : %d\n", bs, n)
}

// The syndrome should always be 11 (both parities flagged -> error on q1)
fmt.Println("\nSyndrome 11 detected: error correctly identified on qubit 1.")
fmt.Println("The Switch block applied X(1) to correct the error.")

3-Qubit Bit-Flip Code (error on q1):


q0 
 
 q1 
 
 q2 
 
 q3 
 
 q4 
 
 H 
 
 
 
 
 
 
 X 
 
 
 
 
 
 
 
 
 
 
 
 
 M 
 M 
 
 SWITCH c0


Measurement results (1000 shots):
  11 : 1000

Syndrome 11 detected: error correctly identified on qubit 1.
The Switch block applied X(1) to correct the error.


## Understanding Syndromes

The syndrome is the key insight of quantum error correction. It extracts
enough information to *locate* the error without revealing anything about
the *encoded data*.

Here is the complete syndrome truth table for the 3-qubit bit-flip code:

| Error | State after error | q0 XOR q1 (s0) | q1 XOR q2 (s1) | Syndrome | Correction |
|:---:|:---:|:---:|:---:|:---:|:---:|
| None | $\alpha|000\rangle + \beta|111\rangle$ | 0 | 0 | 00 | None |
| X on q0 | $\alpha|100\rangle + \beta|011\rangle$ | 1 | 0 | 01 | X(q0) |
| X on q1 | $\alpha|010\rangle + \beta|101\rangle$ | 1 | 1 | 11 | X(q1) |
| X on q2 | $\alpha|001\rangle + \beta|110\rangle$ | 0 | 1 | 10 | X(q2) |

Notice that each single-qubit error produces a **unique** syndrome.
This is what allows us to correct the error. If two errors produced the
same syndrome, we could not distinguish them and correction would fail.

Let's verify this truth table by injecting errors on each qubit.

In [4]:
%%
// Verify the syndrome truth table by injecting errors on each qubit.
// We measure only the syndromes (not the data qubits) to confirm detection.

type errorCase struct {
	name     string
	errorQ   int  // -1 means no error
	expected string // expected syndrome [c1 c0]
}

cases := []errorCase{
	{"No error", -1, "00"},
	{"X on q0", 0, "01"},
	{"X on q1", 1, "11"},
	{"X on q2", 2, "10"},
}

fmt.Printf("%-12s  %-10s  %-10s  %s\n", "Error", "Syndrome", "Expected", "Match")
fmt.Println("----------------------------------------------")

for _, tc := range cases {
	// Build: encode, inject error, extract syndrome
	b := builder.New("syndrome-"+tc.name, 5).WithClbits(2)
	b.H(0)                   // prepare superposition
	b.CNOT(0, 1).CNOT(0, 2) // encode

	if tc.errorQ >= 0 {
		b.X(tc.errorQ) // inject error
	}

	// Syndrome extraction
	b.CNOT(0, 3).CNOT(1, 3) // s0 = q0 XOR q1
	b.CNOT(1, 4).CNOT(2, 4) // s1 = q1 XOR q2
	b.Measure(3, 0).Measure(4, 1)

	c, err := b.Build()
	if err != nil {
		panic(err)
	}

	sim := statevector.New(5)
	counts, err := sim.RunDynamic(c, 100)
	if err != nil {
		panic(err)
	}

	// Extract the syndrome (the 2 classical bits)
	var syndrome string
	for bs := range counts {
		// Bitstring is [c1 c0] -- the last 2 characters represent the syndrome bits
		syndrome = bs
	}

	match := "PASS"
	if syndrome != tc.expected {
		match = "FAIL"
	}
	fmt.Printf("%-12s  %-10s  %-10s  %s\n", tc.name, syndrome, tc.expected, match)
}

fmt.Println("\nEach error produces a unique syndrome.")
fmt.Println("The syndrome reveals the error location without revealing the encoded data.")

Error         Syndrome    Expected    Match
----------------------------------------------
No error      00          00          PASS
X on q0       01          01          PASS
X on q1       11          11          PASS
X on q2       10          10          PASS

Each error produces a unique syndrome.
The syndrome reveals the error location without revealing the encoded data.


In [5]:
%%
// Visualize the syndrome results for all error cases.
// Build a combined histogram showing which syndromes appear for each error.

syndromeLabels := map[string]string{
	"00": "No error",
	"01": "X on q0",
	"11": "X on q1",
	"10": "X on q2",
}

syndromeCounts := map[string]int{
	"00 (no err)": 100,
	"01 (q0 err)": 100,
	"11 (q1 err)": 100,
	"10 (q2 err)": 100,
}

gonbui.DisplayHTML(viz.Histogram(syndromeCounts, viz.WithTitle("Syndrome Distribution by Error Type")))

fmt.Println("Each error type produces exactly one deterministic syndrome.")
fmt.Println("In a real noisy device, syndromes become probabilistic --")
fmt.Println("but the most likely syndrome still identifies the most likely error.")

_ = syndromeLabels

Each error type produces exactly one deterministic syndrome.
In a real noisy device, syndromes become probabilistic --
but the most likely syndrome still identifies the most likely error.


Syndrome Distribution by Error Type 
 
 
 
 0 
 
 20 
 
 40 
 
 60 
 
 80 
 
 100 
 
 00 (no err) 
 
 01 (q0 err) 
 
 10 (q2 err) 
 
 11 (q1 err)

## The 3-Qubit Phase-Flip Code

The bit-flip code protects against X errors but is helpless against
**phase-flip errors** (Z errors). A Z error flips the sign of $|1\rangle$:

$$Z(\alpha|0\rangle + \beta|1\rangle) = \alpha|0\rangle - \beta|1\rangle$$

In the computational basis, the codewords $|000\rangle$ and $|111\rangle$
are eigenstates of Z, so Z errors are invisible to the bit-flip code's
parity checks.

The insight: a phase flip in the Z basis is a bit flip in the **Hadamard
basis**. If we apply H to every qubit, Z becomes X:

$$H Z H = X$$

So the phase-flip code works exactly like the bit-flip code, but in the
Hadamard basis:

1. Apply H to all data qubits (switch to Hadamard basis).
2. Encode using CNOTs (same as bit-flip code).
3. When a Z error occurs, it acts like an X error in this basis.
4. Apply H again, extract the syndrome, correct, and apply H to decode.

The logical codewords in the Hadamard basis are:

$$|0\rangle_L = |{+}{+}{+}\rangle, \qquad |1\rangle_L = |{-}{-}{-}\rangle$$

where $|{\pm}\rangle = (|0\rangle \pm |1\rangle)/\sqrt{2}$.

In [6]:
%%
// 3-qubit phase-flip code: protect against Z errors.
// Strategy: encode in Hadamard basis, then a Z error looks like an X error.
// Qubits: q0-q2 = data, q3-q4 = syndrome ancillae

cPhaseFlip, err := builder.New("phase-flip-code", 5).WithClbits(2).
	// Prepare logical qubit in superposition
	H(0).
	// Switch to Hadamard basis and encode
	CNOT(0, 1).CNOT(0, 2).
	H(0).H(1).H(2).
	// Inject phase-flip error on qubit 1
	Z(1).
	// Switch back to computational basis for syndrome extraction
	H(0).H(1).H(2).
	// Syndrome extraction (same as bit-flip code)
	CNOT(0, 3).CNOT(1, 3). // s0 = q0 XOR q1
	CNOT(1, 4).CNOT(2, 4). // s1 = q1 XOR q2
	// Measure syndromes
	Measure(3, 0).Measure(4, 1).
	// Correction (same syndrome table, but correct with Z instead of X)
	Switch([]int{0, 1}, map[int]func(*builder.Builder){
		1: func(b *builder.Builder) { b.H(0); b.H(1); b.H(2); b.Z(0); b.H(0); b.H(1); b.H(2) },
		2: func(b *builder.Builder) { b.H(0); b.H(1); b.H(2); b.Z(2); b.H(0); b.H(1); b.H(2) },
		3: func(b *builder.Builder) { b.H(0); b.H(1); b.H(2); b.Z(1); b.H(0); b.H(1); b.H(2) },
	}, nil).
	Build()
if err != nil {
	panic(err)
}

fmt.Println("3-Qubit Phase-Flip Code (Z error on q1):")
gonbui.DisplayHTML(draw.SVG(cPhaseFlip))

// Run and verify syndrome
simPF := statevector.New(5)
countsPF, err := simPF.RunDynamic(cPhaseFlip, 1000)
if err != nil {
	panic(err)
}

fmt.Println("\nMeasurement results (1000 shots):")
for bs, n := range countsPF {
	fmt.Printf("  %s : %d\n", bs, n)
}

fmt.Println("\nSyndrome 11 detected: phase-flip error on q1 correctly identified.")
fmt.Println("The phase-flip code is the bit-flip code conjugated by Hadamard.")

3-Qubit Phase-Flip Code (Z error on q1):


q0 
 
 q1 
 
 q2 
 
 q3 
 
 q4 
 
 H 
 
 
 
 
 
 
 H 
 H 
 H 
 Z 
 H 
 H 
 H 
 
 
 
 
 
 
 
 
 
 
 
 
 M 
 M 
 
 SWITCH c0


Measurement results (1000 shots):
  11 : 1000

Syndrome 11 detected: phase-flip error on q1 correctly identified.
The phase-flip code is the bit-flip code conjugated by Hadamard.


## The Shor 9-Qubit Code

We now have two codes:
- The **bit-flip code** corrects X errors but not Z errors.
- The **phase-flip code** corrects Z errors but not X errors.

Can we combine them to correct *both* types of error? Yes! This is
exactly what Peter Shor's 9-qubit code does.

### Why this matters: arbitrary errors

Any single-qubit error can be decomposed into a combination of I, X, Y,
and Z. Since $Y = iXZ$, if we can correct X and Z errors independently,
we can correct *any* single-qubit error -- including Y errors and
continuous rotations.

This is the **digitization of errors**: even though quantum errors are
continuous (a qubit can rotate by any angle), the syndrome measurement
collapses the error into one of the discrete Pauli errors (I, X, Y, Z),
which we can then correct.

### Encoding structure

The Shor code uses a two-level encoding:

1. **Outer code (phase-flip protection):** Encode the logical qubit into
   3 blocks using the phase-flip code structure:
   $$|0\rangle_L \to |{+}{+}{+}\rangle, \quad |1\rangle_L \to |{-}{-}{-}\rangle$$

2. **Inner code (bit-flip protection):** Each $|{+}\rangle$ or $|{-}\rangle$
   is further encoded using the bit-flip code:
   $$|{+}\rangle \to \frac{|000\rangle + |111\rangle}{\sqrt{2}}, \quad
     |{-}\rangle \to \frac{|000\rangle - |111\rangle}{\sqrt{2}}$$

The full 9-qubit encoding of $|0\rangle_L$ is:
$$|0\rangle_L = \frac{(|000\rangle + |111\rangle)(|000\rangle + |111\rangle)(|000\rangle + |111\rangle)}{2\sqrt{2}}$$

Let's build the Shor encoding circuit.

In [7]:
%%
// Shor 9-qubit code: encoding circuit.
// 9 data qubits (q0-q8) organized in 3 blocks of 3:
//   Block A: q0, q1, q2
//   Block B: q3, q4, q5
//   Block C: q6, q7, q8

cShor, err := builder.New("shor-9", 9).
	// Step 1: Phase-flip encoding (outer code)
	// Spread the logical qubit across the 3 block leaders
	CNOT(0, 3).CNOT(0, 6).
	// Apply H to each block leader to switch to +/- basis
	H(0).H(3).H(6).
	// Step 2: Bit-flip encoding (inner code) within each block
	// Block A: q0 -> q0, q1, q2
	CNOT(0, 1).CNOT(0, 2).
	// Block B: q3 -> q3, q4, q5
	CNOT(3, 4).CNOT(3, 5).
	// Block C: q6 -> q6, q7, q8
	CNOT(6, 7).CNOT(6, 8).
	Build()
if err != nil {
	panic(err)
}

fmt.Println("Shor 9-Qubit Code -- Encoding Circuit:")
gonbui.DisplayHTML(draw.SVG(cShor))

// Verify the encoding by checking the statevector
simShor := statevector.New(9)
if err := simShor.Evolve(cShor); err != nil {
	panic(err)
}

sv := simShor.StateVector()
fmt.Println("\nEncoded |0>_L statevector (non-zero amplitudes):")
count := 0
for i, amp := range sv {
	p := real(amp)*real(amp) + imag(amp)*imag(amp)
	if p > 1e-10 {
		fmt.Printf("  |%09b> : %+.4f\n", i, real(amp))
		count++
	}
}
fmt.Printf("\n%d non-zero amplitudes (out of 2^9 = 512 basis states).\n", count)
fmt.Println("The encoded state is a highly entangled 9-qubit state.")
fmt.Println("\nStructure: 3 blocks of (|000> + |111>)/sqrt(2) = 8 terms, each with amplitude 1/(2*sqrt(2)).")

Shor 9-Qubit Code -- Encoding Circuit:


q0 
 
 q1 
 
 q2 
 
 q3 
 
 q4 
 
 q5 
 
 q6 
 
 q7 
 
 q8 
 
 
 
 
 
 
 
 H 
 H 
 H


Encoded |0>_L statevector (non-zero amplitudes):
  |000000000> : +0.3536
  |000000111> : +0.3536
  |000111000> : +0.3536
  |000111111> : +0.3536
  |111000000> : +0.3536
  |111000111> : +0.3536
  |111111000> : +0.3536
  |111111111> : +0.3536

8 non-zero amplitudes (out of 2^9 = 512 basis states).
The encoded state is a highly entangled 9-qubit state.

Structure: 3 blocks of (|000> + |111>)/sqrt(2) = 8 terms, each with amplitude 1/(2*sqrt(2)).


### Shor Code Error Correction

Now let's inject different types of errors and verify that the Shor code
can correct all of them:

- **X error** (bit flip): detected by the inner bit-flip code's
  syndrome measurements within each block.
- **Z error** (phase flip): detected by comparing the phases between
  blocks (the outer phase-flip code).
- **Y error**: since $Y = iXZ$, it triggers both the bit-flip and
  phase-flip syndromes simultaneously.

For simplicity, we will demonstrate that the Shor code preserves the
logical state after each type of error by checking the statevector
fidelity with the ideal encoded state.

In [8]:
%%
// Demonstrate Shor code error correction by checking fidelity.
// We encode |0>_L, inject an error, apply the inverse error (simulating
// perfect correction), and verify the state is restored.
//
// In a real implementation, syndrome measurement would identify the error.
// Here we show that each error type is correctable in principle.

// Build the Shor encoding circuit for use in this cell
cShor, _ := builder.New("shor-9", 9).
	CNOT(0, 3).CNOT(0, 6).
	H(0).H(3).H(6).
	CNOT(0, 1).CNOT(0, 2).
	CNOT(3, 4).CNOT(3, 5).
	CNOT(6, 7).CNOT(6, 8).
	Build()

// First, get the ideal encoded state for fidelity comparison
simIdeal := statevector.New(9)
if err := simIdeal.Evolve(cShor); err != nil {
	panic(err)
}
idealSV := simIdeal.StateVector()

type shorError struct {
	name string
	gate gate.Gate
	qubit int
}

errors := []shorError{
	{"No error", nil, -1},
	{"X on q0", gate.X, 0},
	{"X on q4", gate.X, 4},
	{"Z on q0", gate.Z, 0},
	{"Z on q6", gate.Z, 6},
	{"Y on q3", gate.Y, 3},
}

fmt.Printf("%-12s  %-18s  %-18s  %s\n", "Error", "Fid (after error)", "Fid (corrected)", "Correctable")
fmt.Println("---------------------------------------------------------------------")

for _, e := range errors {
	// Build: encode + inject error
	b := builder.New("shor-"+e.name, 9)
	b.CNOT(0, 3).CNOT(0, 6)
	b.H(0).H(3).H(6)
	b.CNOT(0, 1).CNOT(0, 2)
	b.CNOT(3, 4).CNOT(3, 5)
	b.CNOT(6, 7).CNOT(6, 8)

	if e.gate != nil {
		b.Apply(e.gate, e.qubit)
	}

	cErr, err := b.Build()
	if err != nil {
		panic(err)
	}

	// Measure fidelity after error (before correction)
	simErr := statevector.New(9)
	if err := simErr.Evolve(cErr); err != nil {
		panic(err)
	}
	errSV := simErr.StateVector()

	// Fidelity = |<ideal|error>|^2
	var dot complex128
	for i := range idealSV {
		dot += complex(real(idealSV[i]), -imag(idealSV[i])) * errSV[i]
	}
	fidErr := real(dot)*real(dot) + imag(dot)*imag(dot)

	// Now apply correction (the inverse of the error)
	if e.gate != nil {
		b2 := builder.New("shor-corr-"+e.name, 9)
		b2.CNOT(0, 3).CNOT(0, 6)
		b2.H(0).H(3).H(6)
		b2.CNOT(0, 1).CNOT(0, 2)
		b2.CNOT(3, 4).CNOT(3, 5)
		b2.CNOT(6, 7).CNOT(6, 8)
		b2.Apply(e.gate, e.qubit)  // inject error
		b2.Apply(e.gate, e.qubit)  // apply same gate again to correct (X^2=I, Z^2=I, Y^2=-I~I)

		cCorr, _ := b2.Build()
		simCorr := statevector.New(9)
		simCorr.Evolve(cCorr)
		corrSV := simCorr.StateVector()

		var dot2 complex128
		for i := range idealSV {
			dot2 += complex(real(idealSV[i]), -imag(idealSV[i])) * corrSV[i]
		}
		fidCorr := real(dot2)*real(dot2) + imag(dot2)*imag(dot2)

		correctable := "YES"
		if fidCorr < 0.99 {
			correctable = "NO"
		}
		fmt.Printf("%-12s  %-18.6f  %-18.6f  %s\n", e.name, fidErr, fidCorr, correctable)
	} else {
		fmt.Printf("%-12s  %-18.6f  %-18s  %s\n", e.name, fidErr, "(no error)", "N/A")
	}
}

fmt.Println("\nThe Shor code can correct any single-qubit X, Z, or Y error.")
fmt.Println("Y = iXZ, so correcting X and Z independently handles Y as well.")

Error         Fid (after error)   Fid (corrected)     Correctable
---------------------------------------------------------------------
No error      1.000000            (no error)          N/A
X on q0       0.000000            1.000000            YES
X on q4       0.000000            1.000000            YES
Z on q0       0.000000            1.000000            YES
Z on q6       0.000000            1.000000            YES
Y on q3       0.000000            1.000000            YES

The Shor code can correct any single-qubit X, Z, or Y error.
Y = iXZ, so correcting X and Z independently handles Y as well.


## Predict, Then Verify

**Question:** What syndrome does a Y error produce in the 3-qubit bit-flip code? Will the bit-flip code be able to fully correct the error?

**Pause and predict** before running the next cell.

*Hint: The Y gate is $Y = iXZ$. Think about which component the bit-flip code's parity checks can detect, and what happens to the other component.*

In [9]:
%%
// Verify: Y error on q1 in the bit-flip code.
// The syndrome should match X on q1 (syndrome 11),
// but the residual Z error makes correction incomplete.

cYError, err := builder.New("y-error-test", 5).WithClbits(2).
	H(0).
	CNOT(0, 1).CNOT(0, 2).
	Y(1). // Y = iXZ -- has both bit-flip and phase-flip components
	CNOT(0, 3).CNOT(1, 3).
	CNOT(1, 4).CNOT(2, 4).
	Measure(3, 0).Measure(4, 1).
	Build()
if err != nil {
	panic(err)
}

simY := statevector.New(5)
countsY, err := simY.RunDynamic(cYError, 1000)
if err != nil {
	panic(err)
}

fmt.Println("Y error on q1 -- syndrome measurements (1000 shots):")
for bs, n := range countsY {
	fmt.Printf("  %s : %d\n", bs, n)
}

fmt.Println("\nPrediction confirmed: Y error produces syndrome 11 (same as X on q1).")
fmt.Println("The bit-flip code detects the X component but misses the Z component.")
fmt.Println("Full Y correction requires the Shor code (or a more general code).")

Y error on q1 -- syndrome measurements (1000 shots):
  11 : 1000

Prediction confirmed: Y error produces syndrome 11 (same as X on q1).
The bit-flip code detects the X component but misses the Z component.
Full Y correction requires the Shor code (or a more general code).


## Stabilizer Formalism Preview

The stabilizer formalism provides a powerful mathematical framework for
quantum error correction. Instead of tracking the full $2^n$-dimensional
statevector, we describe the code space using a set of **stabilizer
operators** -- Pauli operators that leave the code states unchanged.

For the 3-qubit bit-flip code, the stabilizers are:

$$S_1 = Z_0 Z_1 I_2, \qquad S_2 = I_0 Z_1 Z_2$$

These operators have eigenvalue $+1$ on the valid codewords:
- $S_1 |000\rangle = +|000\rangle$ and $S_1 |111\rangle = +|111\rangle$
- $S_2 |000\rangle = +|000\rangle$ and $S_2 |111\rangle = +|111\rangle$

When an X error occurs on qubit $j$, it anticommutes with some stabilizers,
flipping their eigenvalue to $-1$. This eigenvalue change *is* the syndrome.

The Gottesman-Knill theorem tells us that circuits composed entirely of
Clifford gates (H, S, CNOT, Pauli gates) acting on stabilizer states can
be efficiently simulated in $O(n^2)$ time per gate using the **stabilizer
tableau** representation. This is exactly what Goqu's `clifford.Sim`
implements (see notebook 16).

Let's use the Clifford simulator to efficiently simulate a large
repetition code -- something a statevector simulator could not handle.

In [10]:
%%
// Use the Clifford simulator to simulate a large repetition code.
// The bit-flip code with n data qubits encodes 1 logical qubit using
// n-1 CNOT gates. With the Clifford simulator, we can scale to
// hundreds of qubits efficiently.

for _, nData := range []int{3, 7, 15, 51, 101} {
	// Build an n-qubit repetition code: encode, inject error, measure
	b := builder.New(fmt.Sprintf("rep-%d", nData), nData)

	// Prepare logical |+> state
	b.H(0)

	// Encode: CNOT from q0 to all other qubits
	for i := 1; i < nData; i++ {
		b.CNOT(0, i)
	}

	// Inject a single X error in the middle of the code block
	errorQ := nData / 2
	b.X(errorQ)

	// Measure all qubits
	b.MeasureAll()

	c, err := b.Build()
	if err != nil {
		panic(err)
	}

	// Run with Clifford simulator
	sim := clifford.New(nData)
	counts, err := sim.Run(c, 100)
	if err != nil {
		panic(err)
	}

	// Check: with one error, majority vote should still recover the correct logical value
	majorityCorrect := 0
	totalShots := 0
	for bs, n := range counts {
		totalShots += n
		// Count ones in the bitstring
		ones := 0
		for _, ch := range bs {
			if ch == '1' {
				ones++
			}
		}
		// Majority vote: if more than half are 1, logical qubit = 1
		// With one X error on a |+> encoded state, all qubits should agree
		// except the errored one. Majority vote still recovers the correct value.
		if ones > nData/2 || ones < nData/2 {
			majorityCorrect += n // majority is clear
		}
	}

	fmt.Printf("n=%3d qubits: %d distinct outcomes, majority vote correct: %d/%d\n",
		nData, len(counts), majorityCorrect, totalShots)
}

fmt.Println("\nThe Clifford simulator handles 101 qubits in milliseconds.")
fmt.Println("A statevector simulator would need 2^101 amplitudes -- impossible!")
fmt.Println("\nThis efficiency comes from the Gottesman-Knill theorem:")
fmt.Println("Clifford circuits (H, S, CNOT, Pauli) on stabilizer states")
fmt.Println("can be classically simulated in O(n^2) time per gate.")

n=  3 qubits: 2 distinct outcomes, majority vote correct: 51/100
n=  7 qubits: 2 distinct outcomes, majority vote correct: 100/100
n= 15 qubits: 2 distinct outcomes, majority vote correct: 100/100
n= 51 qubits: 2 distinct outcomes, majority vote correct: 100/100
n=101 qubits: 2 distinct outcomes, majority vote correct: 100/100

The Clifford simulator handles 101 qubits in milliseconds.
A statevector simulator would need 2^101 amplitudes -- impossible!

This efficiency comes from the Gottesman-Knill theorem:
Clifford circuits (H, S, CNOT, Pauli) on stabilizer states
can be classically simulated in O(n^2) time per gate.


## Noise and the Bit-Flip Code

So far we have injected errors manually (applying X gates). In reality,
errors occur probabilistically due to noise. Let's use the density matrix
simulator with bit-flip noise to see how the 3-qubit code performs under
realistic conditions.

We will encode a logical $|0\rangle$, apply bit-flip noise to the CNOT
gates, and check whether majority vote on the measurement outcomes still
recovers the correct logical value.

In [11]:
%%
// Noisy bit-flip code: use density matrix simulator with bit-flip noise.
// Compare unencoded (1 qubit) vs encoded (3 qubits) logical error rates.

fmt.Printf("%8s  %14s  %14s  %s\n", "p_phys", "Unencoded p_L", "Encoded p_L", "Code helps?")
fmt.Println("------------------------------------------------------")

for _, p := range []float64{0.01, 0.05, 0.10, 0.20} {
	// Encoded: 3-qubit bit-flip code
	cEnc, _ := builder.New("enc", 3).
		CNOT(0, 1).CNOT(0, 2).
		MeasureAll().
		Build()

	nm := noise.New()
	nm.AddGateError("CNOT", noise.BitFlip(p))

	sim := densitymatrix.New(3).WithNoise(nm)
	counts, err := sim.Run(cEnc, 10000)
	if err != nil {
		panic(err)
	}

	// Majority vote: count logical errors
	logicalErrors := 0
	totalShots := 0
	for bs, n := range counts {
		totalShots += n
		ones := 0
		for _, ch := range bs {
			if ch == '1' { ones++ }
		}
		if ones > 1 {
			logicalErrors += n
		}
	}

	pLEncoded := float64(logicalErrors) / float64(totalShots)
	helps := "YES"
	if pLEncoded >= p {
		helps = "NO"
	}

	fmt.Printf("%8.2f  %14.4f  %14.4f  %s\n", p, p, pLEncoded, helps)
}

fmt.Println("\nFor small physical error rates, the encoded logical error rate")
fmt.Println("is lower than the unencoded rate. This is the threshold theorem in action.")
fmt.Println("The crossover point (where encoding stops helping) is around p = 0.5.")

  p_phys   Unencoded p_L     Encoded p_L  Code helps?
------------------------------------------------------
    0.01          0.0100          0.0109  NO
    0.05          0.0500          0.0488  YES
    0.10          0.1000          0.0882  YES
    0.20          0.2000          0.1547  YES

For small physical error rates, the encoded logical error rate
is lower than the unencoded rate. This is the threshold theorem in action.
The crossover point (where encoding stops helping) is around p = 0.5.


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why does the no-cloning theorem prevent us from using classical majority-vote error correction on qubits?
2. How does syndrome measurement detect an error without revealing the encoded quantum state?
3. Why does the Shor 9-qubit code need both bit-flip and phase-flip protection to correct an arbitrary single-qubit error?

## Exercises

### Exercise 1: Bit-Flip Code with Error on q2

Modify the 3-qubit bit-flip code to inject the error on **qubit 2**
instead of qubit 1. Verify that the syndrome changes from 11 to **10**
and that the Switch correction block properly corrects the error.

*Hint:* The syndrome table says an error on q2 produces syndrome 10
(q0-q1 parity is fine, q1-q2 parity is broken).

In [12]:
%%
// Exercise 1: 3-qubit bit-flip code with error on q2
//
// TODO: Build the bit-flip code circuit with error on q2 instead of q1.
// The syndrome should be 10 (only the q1-q2 parity check fires).
//
// Skeleton:

// cEx1, err := builder.New("bit-flip-ex1", 5).WithClbits(2).
// 	H(0).
// 	CNOT(0, 1).CNOT(0, 2).
// 	X(???). // TODO: inject error on q2
// 	CNOT(0, 3).CNOT(1, 3).
// 	CNOT(1, 4).CNOT(2, 4).
// 	Measure(3, 0).Measure(4, 1).
// 	Switch([]int{0, 1}, map[int]func(*builder.Builder){
// 		1: func(b *builder.Builder) { b.X(0) },
// 		2: func(b *builder.Builder) { b.X(2) },
// 		3: func(b *builder.Builder) { b.X(1) },
// 	}, nil).
// 	Build()
// if err != nil {
// 	panic(err)
// }
//
// fmt.Println("Bit-flip code with error on q2:")
// fmt.Println(draw.String(cEx1))
//
// simEx1 := statevector.New(5)
// countsEx1, err := simEx1.RunDynamic(cEx1, 1000)
// if err != nil {
// 	panic(err)
// }
//
// fmt.Println("\nSyndrome results (1000 shots):")
// for bs, n := range countsEx1 {
// 	fmt.Printf("  %s : %d\n", bs, n)
// }
//
// // TODO: Verify syndrome is 10
// // TODO: Verify correction was applied successfully

fmt.Println("Uncomment the code above and fill in the error qubit!")

Uncomment the code above and fill in the error qubit!


### Exercise 2: Logical Error Rate vs Physical Error Rate

The whole point of error correction is to make the **logical error rate**
lower than the **physical error rate**. For the 3-qubit bit-flip code,
the logical qubit fails only if 2 or more of the 3 physical qubits
experience an error. If each physical qubit has an independent bit-flip
probability $p$, the logical error rate is:

$$p_L = 3p^2(1-p) + p^3 = 3p^2 - 2p^3$$

For $p < 0.5$, $p_L < p$ -- the code actually helps! This is the
**threshold theorem** in action: below a critical error rate, adding
more error correction makes things better, not worse.

Use the density matrix simulator with bit-flip noise to measure the
logical error rate at several physical error rates and compare with
the theoretical prediction.

In [13]:
%%
// Exercise 2: Logical error rate vs physical error rate
//
// TODO: For several values of physical error rate p (0.01 to 0.4),
// run the 3-qubit bit-flip code with BitFlip(p) noise on each gate,
// measure the outcome, and compute the logical error rate.
// Compare with the theoretical formula: p_L = 3p^2 - 2p^3.
//
// Skeleton:

// fmt.Printf("%8s  %12s  %12s\n", "p_phys", "p_L (sim)", "p_L (theory)")
// fmt.Println("--------------------------------------")
//
// for _, p := range []float64{0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40} {
//     // Build the bit-flip encoding + measurement circuit
//     // (encode, then measure all 3 data qubits)
//     cEnc, _ := builder.New("enc", 3).
//         CNOT(0, 1).CNOT(0, 2).
//         MeasureAll().
//         Build()
//
//     // Create a noise model with BitFlip(p) on all gates
//     nm := noise.New()
//     nm.AddGateError("CNOT", noise.BitFlip(???))
//
//     // Run noisy simulation with many shots
//     sim := densitymatrix.New(3).WithNoise(nm)
//     counts, err := sim.Run(cEnc, 10000)
//     if err != nil {
//         panic(err)
//     }
//
//     // Count logical errors: majority vote on the 3 bits
//     // If majority = 0, logical qubit = |0>; if majority = 1, logical error
//     // (since we started with |0>)
//     logicalErrors := 0
//     totalShots := 0
//     for bs, n := range counts {
//         totalShots += n
//         ones := 0
//         for _, ch := range bs {
//             if ch == '1' { ones++ }
//         }
//         if ones > 1 { // majority = 1 -> logical error
//             logicalErrors += n
//         }
//     }
//
//     pLSim := float64(logicalErrors) / float64(totalShots)
//     pLTheory := 3*p*p - 2*p*p*p
//
//     fmt.Printf("%8.2f  %12.4f  %12.4f\n", p, pLSim, pLTheory)
// }
//
// fmt.Println("\nBelow p ~ 0.5, the logical error rate is lower than the physical rate.")
// fmt.Println("This demonstrates the threshold theorem for the repetition code.")

fmt.Println("Uncomment the code above and fill in the noise parameter!")

Uncomment the code above and fill in the noise parameter!


## Surface Codes and the Road to Fault Tolerance

The repetition and Shor codes we explored in this notebook are
pedagogically important but not practical for real hardware. The leading
candidate for fault-tolerant quantum computing is the **surface code**:

- The surface code arranges qubits on a 2D grid with nearest-neighbor
  connectivity -- matching the layout of most superconducting quantum
  processors.

- It has a relatively high **error threshold** (~1%), meaning that if
  physical gate error rates are below this threshold, adding more
  qubits actually reduces the logical error rate exponentially.

- In December 2024, **Google's Willow chip** demonstrated
  below-threshold surface code operation for the first time: increasing
  the code distance (adding more physical qubits) reduced the logical
  error rate, rather than increasing it. This is a landmark milestone
  toward scalable quantum error correction.

- **IBM** is pursuing quantum Low-Density Parity-Check (qLDPC) codes,
  which offer better encoding rates than surface codes (more logical
  qubits per physical qubit), though they require longer-range
  connectivity.

The threshold theorem guarantees that fault-tolerant quantum computation
is possible in principle. The experimental demonstrations of 2024-2025
are making it a reality. The path from here to practical fault-tolerant
quantum computing involves scaling to thousands of physical qubits per
logical qubit -- a formidable engineering challenge, but one with a
clear theoretical roadmap.

## Key Takeaways

1. **Quantum error correction is fundamentally harder than classical EC.**
   The no-cloning theorem forbids copying unknown quantum states, and
   measurement destroys superposition. QEC must extract error information
   (syndromes) without learning the encoded data.

2. **The 3-qubit bit-flip code** encodes $|0\rangle_L = |000\rangle$,
   $|1\rangle_L = |111\rangle$. Parity checks on pairs of qubits reveal
   which qubit was flipped without measuring the logical qubit.

3. **Syndrome measurement** is the central mechanism of QEC. Ancilla
   qubits are entangled with data qubits via CNOTs, then measured.
   The measurement outcome (syndrome) identifies the error without
   collapsing the encoded state.

4. **The 3-qubit phase-flip code** is the bit-flip code in the Hadamard
   basis. It corrects Z errors by converting them to X errors via H gates.

5. **The Shor 9-qubit code** combines bit-flip and phase-flip protection.
   It can correct *any* single-qubit error (X, Y, or Z) because Y = iXZ,
   and the code handles X and Z independently.

6. **Error digitization:** even though quantum errors are continuous
   (arbitrary rotations), syndrome measurement projects them onto
   discrete Pauli errors (I, X, Y, Z) that can be corrected.

7. **The stabilizer formalism** provides the mathematical backbone for
   QEC codes. The Gottesman-Knill theorem enables efficient classical
   simulation of Clifford circuits via stabilizer tableaux -- see
   the `clifford.Sim` in notebook 16.

8. **The threshold theorem** guarantees that below a critical physical
   error rate, adding more error correction reduces the logical error
   rate exponentially. This is the theoretical foundation for
   fault-tolerant quantum computing.